## Snakemake for Converting gpep Forcing to Snakemake ##

Generate the snakemake rules used in the conversion of gpep forcing to summa input forcing



### Prep gpep files for processing ###

In [ ]:
#Define config file
config_file = '../config/gpep_to_summa_hydrofabric_config.yaml'

In [2]:
%%writefile ../rules/gpep_file_prep.smk

# This Snakemake file prepares the GPEP  data for use in 
from pathlib import Path
import sys
from pprint import pprint

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils

# Resolve paths from the configuration file
config = gts_utils.resolve_paths(config)
pprint(config)

#Set the list of forcing files to process
# Read all forcing files and create a list based on the output directory (i.e. ens/filename.nc)
ensemble_list, file_path_list = gts_utils.build_ensemble_list(config['gpep_forcing_dir'])

#This first rule establishes the output files that will be created
rule gpep_file_prep:
    input:
        expand(Path(config['gpep_tmp_forcing_dir'],"{forcing_file}.nc"), forcing_file=file_path_list)

#Add greogrian calendar to the time variable, needed for easymore  
rule add_gregorian_to_nc:
    input:  
        input_forcing = Path(config['gpep_forcing_dir'],"{id}.nc")
    output: 
        output_forcing = temp(Path(config['gpep_tmp_forcing_dir'],"{id}_greg.nc"))
    group:
        "gpep_file_prep"
    resources:
        runtime= 1
    shell: 
        """
        mkdir -p $(dirname {output.output_forcing})
        ncatted -a "calendar,time,o,c,"gregorian"" {input.input_forcing} {output.output_forcing}
        """
#Process temperature data to create t_max and t_min
rule add_t_max_and_t_min:
    input: 
        input_file = Path(config['gpep_tmp_forcing_dir'],"{id}_greg.nc")
    output:
        temp1 = temp(Path(config['gpep_tmp_forcing_dir'],"{ens}","{id}_temp1.nc")),
        output_file = Path(config['gpep_tmp_forcing_dir'],"{ens}","{id}.nc")
    params:
        t_mean_var = 'tmean',
        t_range_var = 'trange'
    group:
        "gpep_file_prep"
    resources:
        runtime= 10,
        mem_mb= 10000
    shell:
        """
        # Create directories for output files if they don't exist
        ncap2 -s '
          where({params.t_range_var} < 0) {params.t_range_var} = 0;
          t_max = {params.t_mean_var} + 0.5 * {params.t_range_var};
          t_min = {params.t_mean_var} - 0.5 * {params.t_range_var}
        ' {input.input_file} {output.temp1}

        # Add attributes and rename variables in one step with ncatted and ncrename
        ncatted -O -a long_name,t_max,o,c,"estimated daily maximum temperature" \
                   -a long_name,t_min,o,c,"estimated daily minimum temperature" {output.temp1}
        ncrename -O -v lat,latitude -v lon,longitude {output.temp1} {output.output_file}
        """

Overwriting ../rules/gpep_file_prep.smk


### Run snakemake file

In [ ]:
! snakemake --unlock -s ../rules/gpep_file_prep.smk --configfile {config_file}
! snakemake -s ../rules/gpep_file_prep.smk -c 8 --configfile {config_file}

Unlocking working directory.
Building DAG of jobs...
MissingInputException in rule gpep_file_prep in file /home/dcasson/GitRepos/gpep_to_summa_snakemake/workflow/rules/gpep_file_prep.smk, line 17:
Missing input files for rule gpep_file_prep:
    affected files:
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/lwr_dynamic_gamma_high_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/lwr_dynamic_ecdf_high_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/lwr_dynamic_boxcox_low_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/rf_static_gamma_high_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/rf_dynamic_ecdf_high_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/lwr_dynamic_ecdf_low_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gpep_tmp/temp2/ors/rf_dynamic_boxcox_low_predictors.nc
        /Users/drc858/Data/hydrofabric/test/gp

## Remap gpep to SHP ##

In [3]:
%%writefile ../rules/remap_gpep_to_shp.smk

# Import needed packages
from pathlib import Path
import sys
sys.path.append('../')
from scripts import remap_forcing_to_shp
from scripts import gpep_to_summa_utils as gts_utils

# Resolve paths from the configuration file
config = gts_utils.resolve_paths(config)

# Read all forcing files and create a list based on the output directory (i.e. ens/filename.nc)
ensemble_list, file_path_list = gts_utils.build_ensemble_list(config['gpep_forcing_dir'])

rule remap_gpep_to_shp:
    input:
        expand(Path(config['easymore_output_dir'],"{file}.nc"), file=file_path_list)

# Define rule to run file remapping when remap file exists
rule create_remap_file:
    input:
        input_forcing_files = expand(Path(config['gpep_tmp_forcing_dir'],"{forcing_file}.nc"), forcing_file=file_path_list),
        input_shp = config['catchment_shp']
    output:
        remap_nc = config['remap_file']
    resources:
        runtime= 2,
        mem_mb= 500
    run:
        remap_forcing_to_shp.remap_with_easymore(config, input.input_forcing_files[0],input.input_shp, output.remap_nc, only_create_remap_nc=True)

# Define rule to run file remapping when remap file exists
rule remap_with_easymore:
    input:
        input_forcing = Path(config['gpep_tmp_forcing_dir'],"{id}.nc"),
        input_shp = config['catchment_shp'],
        remap_nc = config['remap_file'],
    output:
        output_forcing = Path(config['easymore_output_dir'],"{id}.nc")
    params:
        file_path = "{id}"
    resources:
        runtime= 120,
        mem_mb= 3000
    run:
        try:
            remap_forcing_to_shp.remap_with_easymore(config, input.input_forcing, input.input_shp, input.remap_nc, only_create_remap_nc=False, file_path=params.file_path)
        except Exception as e:
            # Log the error or take some other action
            print(f"Error occurred: {e}")
            # Optionally, raise the exception to stop the workflow
            raise e


Overwriting ../rules/remap_gpep_to_shp.smk


In [2]:
! snakemake --unlock -s ../rules/remap_gpep_to_shp.smk  --configfile {config_file}
! snakemake -s ../rules/remap_gpep_to_shp.smk -c 8 --configfile {config_file}

Traceback (most recent call last):
  File "/home/dcasson/.local/bin/snakemake", line 8, in <module>
    sys.exit(main())
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 3141, in main
    success = snakemake(
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 570, in snakemake
    update_config(overwrite_config, load_configfile(f))
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/io.py", line 1720, in load_configfile
    config = _load_configfile(configpath)
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/io.py", line 1694, in _load_configfile
    obj = open(configpath_or_obj, encoding="utf-8")
FileNotFoundError: [Errno 2] No such file or directory: '{config_file}'
Traceback (most recent call last):
  File "/home/dcasson/.local/bin/snakemake", line 8, in <module>
    sys.exit(main())
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 3141,

## Prepare gpep files for metsim ##

In [39]:
%%writefile ../rules/metsim_file_prep.smk
from pathlib import Path

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils
from scripts import metsim_utils as ms_utils

# Resolve paths from the configuration file
config = gts_utils.resolve_paths(config)

# Create a list of the forcing files produced in the last workflow
input_forcing_list = gts_utils.list_files_in_subdirectory(config['easymore_output_dir'])
ensemble_list, file_path_list = gts_utils.build_ensemble_list(config['gpep_forcing_dir'])

easymore_output = Path(config['easymore_output_dir']) 
metsim_input = Path(config['metsim_input_dir'])

# Main rule to define the files produced by this workflow
rule metsim_file_prep:
    input:
        Path(config["metsim_dir"], config["metsim_domain_nc"]),
        expand(Path(metsim_input,"{forcing}.nc"), forcing = input_forcing_list),
        expand(Path(metsim_input,"{forcing}_state.nc"), forcing = input_forcing_list)
         
# Create metsim domain file from an existing summa attribute file
rule create_metsim_domain_summa_attr:
    input:
        attr_nc = Path(config["attribute_nc"])
    output:
        domain_nc = Path(config["metsim_dir"], config["metsim_domain_nc"])
    group:
        "prep_metsim"
    resources:
        runtime= 2,
        mem_mb= 500
    shell:
        'ncap2 -O -s "mask=elevation*0+1" {input.attr_nc} {output.domain_nc}'

"""
rule subset_metsim_domain_to_forcing:
    input:
        domain_nc = Path(config["summa_dir"], config["metsim_domain_nc"]),
        input_forcing_files = expand(Path(config['easymore_output_dir'],"{file}.nc"), file=file_path_list)
    output:
        subset_nc = Path(config["metsim_dir"], config["metsim_domain_nc"])
    run:
        ms_utils.subset_domain_to_forcing(input.domain_nc, input.input_forcing_files[0], output.subset_nc)
"""

# Define rule to run file remapping when remap file exists
rule prep_forcing_files_with_hru_id:
    input:
        input_forcing = Path(easymore_output,"{forcing}.nc")
    output:
        hru_id_temp = temp(Path(easymore_output,"{forcing}_hruId.nc")),
        hru_id = Path(metsim_input,"{forcing}.nc")
    group:
        "prep_metsim"
    resources:
        runtime= 2,
        mem_mb= 500
    shell:
        """
        ncrename -O -v .ID,hruId {input.input_forcing}
        ncrename -d .ID,hru {input.input_forcing}
        ncap2 -O -s "hru=array(0,1,hruId)" {input.input_forcing} {output.hru_id_temp};
        ncatted -O -a long_name,hru,a,c,"hru coordinate index" {output.hru_id_temp} {output.hru_id}
        """
        
rule create_state_file:
    input:
        input_forcing_file = Path(metsim_input,"{forcing}.nc")
    output:
        output_state_file = temp(Path(metsim_input,"{forcing}_state_temp.nc"))
    group:
        "prep_metsim"
    resources:
        runtime=2,
        mem_mb= 500
    run:
        ms_utils.create_state_file(input.input_forcing_file, output.output_state_file)

rule update_state_file_time:
    input:
        input_state_file = Path(metsim_input,"{forcing}_state_temp.nc")
    output:
        output_state_file = Path(metsim_input,"{forcing}_state.nc")
    group:
        "prep_metsim"
    resources:
        runtime=2,
        mem_mb=500
    run:
        gts_utils.update_time_units(input.input_state_file, output.output_state_file)


Overwriting ../rules/metsim_file_prep.smk


In [ ]:
! snakemake --unlock -s ../rules/metsim_file_prep.smk  --configfile {config_file}
! snakemake -s ../rules/metsim_file_prep.smk -c 8 --configfile {config_file}

## Run metsim ##

In [9]:
%%writefile ../rules/run_metsim.smk
from pathlib import Path

sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils
from scripts import metsim_utils as ms_utils

# Resolve paths from the configuration file
config = gts_utils.resolve_paths(config)


# Generate list of forcing files to create wildcards
input_forcing_list = gts_utils.list_files_in_subdirectory(Path(config['easymore_output_dir']))

rule run_metsim:
    input:
        expand(Path(config['metsim_output_dir'],"{id}.nc"), id=input_forcing_list)

rule update_metsim_base_time:
    input:
        metsim_input_forcing = Path(config['metsim_input_dir'],"{id}.nc")
    output:
        metsim_temp_input_forcing = temp(Path(config['metsim_input_dir'],'temp',"{id}_input.nc"))
    group:
        "run_metsim"
    resources:
        runtime= 2,
        mem_mb= 500
    run:
        gts_utils.update_time_units(input.metsim_input_forcing, output.metsim_temp_input_forcing)

rule generate_metsim_output:
    input:
        metsim_input_forcing = Path(config['metsim_input_dir'],'temp',"{id}_input.nc"),
        metsim_input_state = Path(config['metsim_input_dir'],"{id}_state.nc"),
        metsim_input_domain = Path(config["metsim_dir"], config["metsim_domain_nc"])
    output:
        metsim_output_forcing = Path(config['metsim_output_dir'],"{id}.nc")
    group:
        "run_metsim"
    resources:
        runtime= 30,
        mem_mb=10000
    run:
        ms = ms_utils.create_metsim_config(config, input.metsim_input_forcing,input.metsim_input_state,output.metsim_output_forcing)
        ms.run()
        ms_utils.rename_metsim_output(ms,output.metsim_output_forcing)



Overwriting ../rules/run_metsim.smk


In [33]:
! snakemake --unlock -s ../rules/run_metsim.smk  --configfile {config_file}
! snakemake -s ../rules/run_metsim.smk -c 8 --configfile {config_file}

Traceback (most recent call last):
  File "/home/dcasson/.local/bin/snakemake", line 8, in <module>
    sys.exit(main())
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 3141, in main
    success = snakemake(
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 570, in snakemake
    update_config(overwrite_config, load_configfile(f))
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/io.py", line 1720, in load_configfile
    config = _load_configfile(configpath)
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/io.py", line 1694, in _load_configfile
    obj = open(configpath_or_obj, encoding="utf-8")
FileNotFoundError: [Errno 2] No such file or directory: '{config_file}'
Traceback (most recent call last):
  File "/home/dcasson/.local/bin/snakemake", line 8, in <module>
    sys.exit(main())
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 3141,

## Convert metsim output to summa ready input ##

In [4]:
%%writefile ../rules/metsim_to_summa.smk
from pathlib import Path

# Import custom functions
sys.path.append('../')
from scripts import gpep_to_summa_utils as gts_utils

# Resolve paths from the configuration file
config = gts_utils.resolve_paths(config)

input_forcing_list = gts_utils.list_files_in_subdirectory(config['metsim_output_dir'], '.nc')

rule metsim_to_summa:
    input:
        expand(Path(config['summa_forcing_dir'],"{id}.nc"), id=input_forcing_list)

rule create_hru_id_file:
    input:
        subset_domain_file = Path(config["metsim_dir"], config["metsim_domain_nc"])
    output:
        hru_id_file = temp(Path(config["metsim_dir"], 'hruId.nc'))
    group:
        "metsim_to_summa"
    resources:
        runtime= 5,
        mem_mb= 2000
    shell:
        'ncks -v hruId {input.subset_domain_file} {output.hru_id_file}'

rule append_hru_id_and_datastep_to_metsim_output:
    input:
        input_metsim_file = Path(config['metsim_output_dir'],"{id}.nc"),
        hru_id_file = Path(config["metsim_dir"], 'hruId.nc')
    output:
        temp_metsim_file = temp(Path(config['metsim_dir'],"temp","{id}.nc")),
        output_metsim_file = Path(config['summa_forcing_dir'],"{id}.nc")
    params:
        timestep = int(config["metsim_timestep_minutes"]) * 60
    group:
        "metsim_to_summa"
    resources:
        runtime= 5,
        mem_mb= 2000
    shell:
        """
        temp_file={output.temp_metsim_file}
        mkdir -p `dirname $temp_file`
        if ! ncdump -h {input.input_metsim_file} | grep -q "hruId"; then
            ncks -h -A {input.hru_id_file} {input.input_metsim_file}
        fi
        ncks -O -C -x -v hru {input.input_metsim_file} $temp_file
        ncrename -O -v .SWradAtm,SWRadAtm $temp_file
        ncrename -O -v .LWradAtm,LWRadAtm $temp_file
        ncap2 -s "data_step={params.timestep}" $temp_file --append {output.output_metsim_file}
        """

        

Overwriting ../rules/metsim_to_summa.smk


In [3]:
! snakemake --unlock -s ../rules/metsim_to_summa.smk  --configfile {config_file}
! snakemake -s ../rules/metsim_to_summa.smk -c 8 --configfile {config_file}

Traceback (most recent call last):
  File "/home/dcasson/.local/bin/snakemake", line 8, in <module>
    sys.exit(main())
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 3141, in main
    success = snakemake(
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 570, in snakemake
    update_config(overwrite_config, load_configfile(f))
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/io.py", line 1720, in load_configfile
    config = _load_configfile(configpath)
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/io.py", line 1694, in _load_configfile
    obj = open(configpath_or_obj, encoding="utf-8")
FileNotFoundError: [Errno 2] No such file or directory: '{config_file}'
Traceback (most recent call last):
  File "/home/dcasson/.local/bin/snakemake", line 8, in <module>
    sys.exit(main())
  File "/home/dcasson/.local/lib/python3.10/site-packages/snakemake/__init__.py", line 3141,